In [2]:
%pip install transformers torch scikit-learn pandas

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ------------------------------- -------- 10.2/12.8 MB 75.1 MB/s eta 0:00:01
   ---------------------------------------- 12.8/12.8 MB 68.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 75.1 MB/s  0:00:00

   ---------------------------------------- 0/3 [safetensors]
   ------------- -------------------------- 1/3 [tokenizers]
   ------------- -------------------------- 1/3 [tokenizers]
   -------------------------- ------------- 2/3 [transformers]
   -------------------------- ------------- 2/3 [transformers]
   -------------------------- ------------- 2/3 [transformers]
   -------------------------- ------------- 2/3 [transformers]
   -------------------------- ------------- 2/3 [transformers]
   -------------------------- ------------- 2/3 [trans


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\solut\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Module 04: AI in Finance - Financial News Insight & Sentiment Application
**Author:** Sergio Sanchez  
**Course:** 6271-ITAI-2372-Artificial Intel Applications  
**Topic:** Traditional NLP Sentiment Classification & Transformer-Based Summarization  

---

## 1. Executive Summary & Business Problem
Financial analysts and portfolio managers ingest thousands of market headlines, earnings call transcripts, and press releases daily. Manually parsing this volume causes analysis latency, delayed response to market-moving events, and inconsistent sentiment interpretation across coverage teams.

### The Objective
1. **Curate & Preprocess:** Ingest domain-specific financial news data and clean textual noise.
2. **Traditional ML Sentiment Engine:** Train a TF-IDF vectorizer paired with a Multinomial Naive Bayes classifier on labeled financial statements.
3. **Transformer Summarization & Insight Engine:** Deploy a local Hugging Face transformer model to extract key takeaways from market reports.
4. **Interactive Inference Test Suite:** Evaluate unstructured market reports to produce an automated sentiment score alongside actionable executive summaries.

## 2. Environment Setup & Library Initialization
We import our data science stack (`pandas`, `re`), classical machine learning components (`scikit-learn`), and the Transformer pipeline (`transformers`).

In [1]:
import re
import string
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from transformers import pipeline

print("Environment modules loaded successfully.")

C:\Users\solut\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1017 15:10:06.953000 6432 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Environment modules loaded successfully.


## 3. Financial Dataset & Preprocessing Pipeline
We construct an annotated dataset reflecting market movements across three sentiment classes: `Positive`, `Negative`, and `Neutral`. 

### Preprocessing Operations:
1. Stripping URLs and web links.
2. Removing punctuation and numeric noise.
3. Normalizing text to lowercase for consistent TF-IDF feature extraction.

In [10]:
# Expanded Representative Financial Headline Dataset
raw_financial_data = {
    "text": [
        "Company reported record quarterly revenue surging 25% year-over-year beating analyst expectations.",
        "Operating profit expanded significantly driven by strong enterprise cloud adoption and cost controls.",
        "Shares rallied 8% in premarket trading following dividend increase and stock buyback announcement.",
        "Semiconductor manufacturer announces groundbreaking AI chip architecture delivering 3x power efficiency.",
        "Firm delivers exceptional margin expansion with strong revenue growth across all units.",
        "Net income plummeted 40% due to soaring supply chain overhead and weakening consumer demand.",
        "Regulatory probe launched into accounting irregularities, triggering a sharp selloff in company stock.",
        "Credit rating downgraded to junk status amid liquidity crunch and looming debt maturities.",
        "Retailer warns of severe margin compression and lowers full-year guidance citing persistent inflation.",
        "Retailer reported declining operating profit, mounting debt obligations, store closures, and workforce reductions.",
        "Federal Reserve maintains interest rate benchmark unchanged at policy meeting as inflation stabilizes.",
        "Board of directors schedules annual shareholder meeting for next month to vote on proxy proposals.",
        "Quarterly earnings release date finalized for next Tuesday following market close.",
        "Central bank updates regulatory capital requirement guidelines for commercial banking institutions."
    ],
    "label": [
        "Positive", "Positive", "Positive", "Positive", "Positive",
        "Negative", "Negative", "Negative", "Negative", "Negative",
        "Neutral", "Neutral", "Neutral", "Neutral"
    ]
}

df_finance = pd.DataFrame(raw_financial_data)

def clean_text(text: str) -> str:
    """
    Cleans financial text by removing URLs, punctuation, and normalizing casing.
    """
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = text.lower().strip()
    return text

df_finance["cleaned_text"] = df_finance["text"].apply(clean_text)
print(f"Dataset successfully compiled: {len(df_finance)} labeled financial entries.")
df_finance[["cleaned_text", "label"]].tail()

Dataset successfully compiled: 14 labeled financial entries.


,cleaned_text,label
9,retailer reported declining operating profit m...,Negative
10,federal reserve maintains interest rate benchm...,Neutral
11,board of directors schedules annual shareholde...,Neutral
12,quarterly earnings release date finalized for ...,Neutral
13,central bank updates regulatory capital requir...,Neutral


## 4. Traditional ML Sentiment Classification
We use a **TF-IDF (Term Frequency-Inverse Document Frequency)** vectorizer to map word importance scores across documents, then train a **Multinomial Naive Bayes (`MultinomialNB`)** classifier to predict sentiment polarity.

In [11]:
# Feature extraction across the compiled corpus
X = df_finance["cleaned_text"]
y = df_finance["label"]

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(X)

# Train Classifier on full representative vocabulary
sentiment_classifier = MultinomialNB()
sentiment_classifier.fit(X_tfidf, y)

# Self-consistency evaluation
y_pred = sentiment_classifier.predict(X_tfidf)
print(f"Model Training Accuracy: {accuracy_score(y, y_pred) * 100:.1f}%\n")
print("Classification Report:")
print(classification_report(y, y_pred, zero_division=0))

def predict_financial_sentiment(text: str) -> str:
    cleaned = clean_text(text)
    transformed = vectorizer.transform([cleaned])
    return sentiment_classifier.predict(transformed)[0]

Model Training Accuracy: 100.0%

Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00         5
     Neutral       1.00      1.00      1.00         4
    Positive       1.00      1.00      1.00         5

    accuracy                           1.00        14
   macro avg       1.00      1.00      1.00        14
weighted avg       1.00      1.00      1.00        14



## 5. Transformer Integration for Financial Summarization
While traditional NLP handles classification, generative LLM transformers excel at contextual synthesis. We load a lightweight Hugging Face transformer model (`t5-small`) using the `pipeline` API to generate concise executive summaries directly from verbose financial articles.

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load tokenizer and sequence-to-sequence model directly
print("Loading t5-small model and tokenizer...")
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Transformer summarizer loaded successfully.")

# 2. Define the robust generation function
def generate_financial_summary(article_text: str) -> str:
    """
    Generates a concise financial summary using the local T5 seq2seq model.
    """
    try:
        input_text = "summarize: " + article_text.strip()
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
        
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=50,
            min_length=15,
            num_beams=2,
            early_stopping=True
        )
        return tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    except Exception as e:
        return f"Summarization failed: {e}"

Loading t5-small model and tokenizer...


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 586.01it/s]

Transformer summarizer loaded successfully.


## 6. End-to-End Application Testing & Validation
We combine the traditional ML sentiment classifier with the transformer summarization engine to analyze complex financial articles.

In [13]:
test_articles = [
    {
        "title": "Alpha Tech Releases Q3 Financial Results",
        "body": "Alpha Tech surged past expectations this quarter as enterprise artificial intelligence solutions grew over 45 percent. Operating margins reached unprecedented highs and the chief financial officer confirmed a raised outlook for fiscal year 2027 while approving a multi-billion dollar stock repurchase plan."
    },
    {
        "title": "Continental Retailers Faces Severe Margin Compression",
        "body": "Continental Retailers reported declining operating profit and mounting debt obligations as consumer discretionary spending dropped sharply across all regions. The executive board announced store closures, workforce reductions, and suspended dividend distributions to maintain short-term liquidity."
    }
]

print("=" * 85)
print("FINANCIAL NEWS INSIGHT & SENTIMENT ANALYSIS ENGINE")
print("=" * 85)

for idx, article in enumerate(test_articles, start=1):
    sentiment = predict_financial_sentiment(article["body"])
    summary = generate_financial_summary(article["body"])
    
    print(f"\n[REPORT {idx}] {article['title']}")
    print("-" * 85)
    print(f"• Input Passage      : {article['body']}")
    print(f"• Classified Sentiment: [{sentiment.upper()}]")
    print(f"• Generated Summary  : {summary}")

print("\n" + "=" * 85)

FINANCIAL NEWS INSIGHT & SENTIMENT ANALYSIS ENGINE

[REPORT 1] Alpha Tech Releases Q3 Financial Results
-------------------------------------------------------------------------------------
• Input Passage      : Alpha Tech surged past expectations this quarter as enterprise artificial intelligence solutions grew over 45 percent. Operating margins reached unprecedented highs and the chief financial officer confirmed a raised outlook for fiscal year 2027 while approving a multi-billion dollar stock repurchase plan.
• Classified Sentiment: [POSITIVE]
• Generated Summary  : the chief financial officer confirms a raised outlook for fiscal year 2027.

[REPORT 2] Continental Retailers Faces Severe Margin Compression
-------------------------------------------------------------------------------------
• Input Passage      : Continental Retailers reported declining operating profit and mounting debt obligations as consumer discretionary spending dropped sharply across all regions. The executiv

## 7. Business Use Case, Ethical Considerations, & Governance

### Business Value Proposition
- **High-Throughput Triage:** Automates screening of thousands of daily financial wires, surfacing high-risk negative news or market-moving positive earnings reports in sub-second latency.
- **Workflow Efficiency:** Reduces analyst briefing review times by consolidating lengthy disclosures into automated bullet points.

### System Limitations & Responsible AI Safeguards
1. **Financial Disclaimer:** This software is a technological proof-of-concept for educational demonstration. It is not licensed financial advisory software and must never be used as the sole basis for capital allocation.
2. **Contextual Inversion & Sarcasm:** Classical bag-of-words and n-gram models can misclassify complex phrases (e.g., "declined to fall" vs. "declined sharply"). Production deployments require financial domain foundation models (e.g., FinBERT).
3. **Data Quality & Drift:** Macroeconomic sentiment shifts during recessions and market shocks, requiring scheduled retraining and continuous validation.